In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    avg,
    count,
    countDistinct,
    rank,
    row_number,
    lag,
    round,
    trim,
    split,
    explode,
    when,
    lit,
    current_date,
    date_sub,
    concat_ws,
    lower,
    regexp_replace
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ============================================================
# 1. SPARK SESSION
# ============================================================

spark = SparkSession.builder \
    .appName("IndianStartupFunding_Gold") \
    .getOrCreate()

print("Spark Session Started")


# ============================================================
# 2. STORAGE PATHS
# ============================================================

storage_account = "startupfundingstorage"

silver_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
    "startup_funding"
)

gold_base_path = (
    f"abfss://gold@{storage_account}.dfs.core.windows.net/"
)

top_sectors_path = gold_base_path + "top_funded_sectors"

city_rank_path = gold_base_path + "city_funding_rank"

sector_yoy_path = gold_base_path + "sector_yoy_snapshot"

investor_count_path = gold_base_path + "investor_deal_count"

avg_stage_path = gold_base_path + "avg_deal_by_stage"


# ============================================================
# 3. READ SILVER DATA
# ============================================================

print("Reading Silver Delta data...")

silver_df = spark.read \
    .format("delta") \
    .load(silver_path)

print("Silver Row Count:", silver_df.count())

print("Silver Columns:")
print(silver_df.columns)

silver_df.printSchema()

display(silver_df)


# ============================================================
# 4. BASIC DATA PREPARATION
# ============================================================

df = silver_df

if "amount_usd" in df.columns:
    df = df.filter(
        col("amount_usd").isNotNull()
    )

if "industry_vertical" in df.columns:
    df = df.withColumn(
        "industry_vertical",
        trim(col("industry_vertical"))
    )

if "city" in df.columns:
    df = df.withColumn(
        "city",
        trim(col("city"))
    )

if "investment_type" in df.columns:
    df = df.withColumn(
        "investment_type",
        trim(col("investment_type"))
    )

if "investor_names" in df.columns:
    df = df.withColumn(
        "investor_names",
        trim(col("investor_names"))
    )


# ============================================================
# 5. TOP FUNDED SECTORS
# ============================================================
#
# Business Question:
# Which sectors attracted the highest cumulative investment?
#
# SQL Technique:
# GROUP BY + SUM + ORDER BY
#
# PDF Requirement:
# top_funded_sectors
# ============================================================

print("Creating top_funded_sectors...")

top_funded_sectors = df.filter(
    col("industry_vertical").isNotNull()
    & (trim(col("industry_vertical")) != "")
).groupBy(
    "industry_vertical"
).agg(
    spark_sum("amount_usd").alias("total_funding_usd"),
    count("*").alias("deal_count"),
    avg("amount_usd").alias("average_deal_usd")
).withColumn(
    "total_funding_usd",
    round(col("total_funding_usd"), 2)
).withColumn(
    "average_deal_usd",
    round(col("average_deal_usd"), 2)
).orderBy(
    col("total_funding_usd").desc()
)

print("Top Funded Sectors:")
display(top_funded_sectors)


# ============================================================
# 6. WRITE TOP FUNDED SECTORS
# ============================================================

top_funded_sectors.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(top_sectors_path)

print("top_funded_sectors written successfully.")


# ============================================================
# 7. CITY FUNDING RANK
# ============================================================
#
# Business Question:
# Which cities are emerging as startup hubs?
#
# SQL Technique:
# RANK() OVER (ORDER BY total_funding)
#
# PDF Requirement:
# city_funding_rank
# ============================================================

print("Creating city_funding_rank...")

city_funding = df.filter(
    col("city").isNotNull()
    & (trim(col("city")) != "")
).groupBy(
    "city"
).agg(
    spark_sum("amount_usd").alias("total_funding_usd"),
    count("*").alias("deal_count")
)

city_window = Window.orderBy(
    col("total_funding_usd").desc()
)

city_funding_rank = city_funding.withColumn(
    "funding_rank",
    rank().over(city_window)
).withColumn(
    "total_funding_usd",
    round(col("total_funding_usd"), 2)
).orderBy(
    col("funding_rank")
)

print("City Funding Ranking:")
display(city_funding_rank)


# ============================================================
# 8. WRITE CITY FUNDING RANK
# ============================================================

city_funding_rank.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(city_rank_path)

print("city_funding_rank written successfully.")


# ============================================================
# 9. SECTOR YEARLY FUNDING
# ============================================================

print("Creating yearly sector funding data...")

sector_yearly = df.filter(
    col("industry_vertical").isNotNull()
    & col("funding_year").isNotNull()
).groupBy(
    "industry_vertical",
    "funding_year"
).agg(
    spark_sum("amount_usd").alias("total_funding_usd"),
    count("*").alias("deal_count")
)

sector_yearly = sector_yearly.withColumn(
    "total_funding_usd",
    round(col("total_funding_usd"), 2)
)

sector_window = Window.partitionBy(
    "industry_vertical"
).orderBy(
    "funding_year"
)

sector_yearly = sector_yearly.withColumn(
    "previous_year_funding_usd",
    lag("total_funding_usd").over(sector_window)
)

sector_yearly = sector_yearly.withColumn(
    "yoy_change_usd",
    round(
        col("total_funding_usd")
        - col("previous_year_funding_usd"),
        2
    )
)

sector_yearly = sector_yearly.withColumn(
    "yoy_change_percentage",
    when(
        col("previous_year_funding_usd").isNull(),
        None
    ).when(
        col("previous_year_funding_usd") == 0,
        None
    ).otherwise(
        round(
            (
                (
                    col("total_funding_usd")
                    - col("previous_year_funding_usd")
                )
                / col("previous_year_funding_usd")
            ) * 100,
            2
        )
    )
)

print("Sector Yearly Funding:")
display(sector_yearly)


# ============================================================
# 10. SCD TYPE 2 - SECTOR YOY SNAPSHOT
# ============================================================
#
# PDF Requirement:
# sector_yoy_snapshot
#
# Historical tracking using:
# MERGE
# SCD Type 2
# CTE / Window logic
#
# ============================================================

print("Preparing SCD Type 2 sector snapshot...")

sector_snapshot = sector_yearly.select(
    "industry_vertical",
    "funding_year",
    "total_funding_usd",
    "previous_year_funding_usd",
    "yoy_change_usd",
    "yoy_change_percentage",
    "deal_count"
)

sector_snapshot = sector_snapshot.withColumn(
    "effective_from",
    current_date()
).withColumn(
    "effective_to",
    lit(None).cast("date")
).withColumn(
    "is_current",
    lit(True)
)

sector_snapshot = sector_snapshot.withColumn(
    "record_hash",
    regexp_replace(
        concat_ws(
            "||",
            col("industry_vertical"),
            col("funding_year"),
            col("total_funding_usd"),
            col("previous_year_funding_usd"),
            col("yoy_change_usd"),
            col("yoy_change_percentage"),
            col("deal_count")
        ),
        "null",
        ""
    )
)

print("Current Sector Snapshot:")
display(sector_snapshot)


# ============================================================
# 11. CREATE SCD2 TABLE IF IT DOES NOT EXIST
# ============================================================

if not DeltaTable.isDeltaTable(
    spark,
    sector_yoy_path
):

    print("SCD2 table does not exist.")
    print("Creating initial SCD2 snapshot...")

    sector_snapshot.write \
        .format("delta") \
        .mode("overwrite") \
        .save(sector_yoy_path)

    print("Initial SCD2 table created.")

else:

    print("Existing SCD2 table found.")
    print("Applying SCD Type 2 logic...")

    sector_delta = DeltaTable.forPath(
        spark,
        sector_yoy_path
    )

    current_records = sector_delta.toDF().filter(
        col("is_current") == True
    )

    changed_records = sector_snapshot.alias("new").join(
        current_records.alias("old"),
        (
            (col("new.industry_vertical") == col("old.industry_vertical"))
            &
            (col("new.funding_year") == col("old.funding_year"))
            &
            (col("old.is_current") == True)
        ),
        "inner"
    ).filter(
        col("new.record_hash") != col("old.record_hash")
    ).select(
        col("new.industry_vertical").alias("industry_vertical"),
        col("new.funding_year").alias("funding_year")
    )

    changed_records.createOrReplaceTempView(
        "changed_sector_records"
    )

    sector_delta.alias("target").merge(
        changed_records.alias("source"),
        """
        target.industry_vertical = source.industry_vertical
        AND target.funding_year = source.funding_year
        AND target.is_current = true
        """
    ).whenMatchedUpdate(
        set={
            "effective_to": "current_date()",
            "is_current": "false"
        }
    ).execute()

    current_after_update = sector_delta.toDF().filter(
        col("is_current") == True
    )

    new_records = sector_snapshot.alias("new").join(
        current_after_update.alias("old"),
        (
            (col("new.industry_vertical") == col("old.industry_vertical"))
            &
            (col("new.funding_year") == col("old.funding_year"))
            &
            (col("old.is_current") == True)
            &
            (col("new.record_hash") == col("old.record_hash"))
        ),
        "left"
    ).filter(
        col("old.industry_vertical").isNull()
    ).select(
        "new.*"
    )

    new_records.write \
        .format("delta") \
        .mode("append") \
        .save(sector_yoy_path)

    print("SCD Type 2 MERGE completed.")


# ============================================================
# 12. READ SCD2 TABLE
# ============================================================

sector_yoy_snapshot = spark.read \
    .format("delta") \
    .load(sector_yoy_path)

print("Sector YoY Snapshot:")
display(
    sector_yoy_snapshot.orderBy(
        "industry_vertical",
        "funding_year",
        "effective_from"
    )
)


# ============================================================
# 13. INVESTOR DEAL COUNT
# ============================================================
#
# Business Question:
# Who are the most active investors by volume?
#
# SQL Technique:
# JOIN + GROUP BY + COUNT
#
# PDF Requirement:
# investor_deal_count
# ============================================================

print("Creating investor_deal_count...")

investor_df = df.filter(
    col("investor_names").isNotNull()
    & (trim(col("investor_names")) != "")
    & (lower(trim(col("investor_names"))) != "unknown")
)

investor_df = investor_df.withColumn(
    "investor",
    explode(
        split(
            col("investor_names"),
            ","
        )
    )
)

investor_df = investor_df.withColumn(
    "investor",
    trim(col("investor"))
)

investor_df = investor_df.filter(
    col("investor").isNotNull()
    & (col("investor") != "")
)

investor_deal_count = investor_df.groupBy(
    "investor"
).agg(
    count("*").alias("deal_count"),
    spark_sum("amount_usd").alias("total_investment_usd"),
    avg("amount_usd").alias("average_deal_usd")
).withColumn(
    "total_investment_usd",
    round(col("total_investment_usd"), 2)
).withColumn(
    "average_deal_usd",
    round(col("average_deal_usd"), 2)
).orderBy(
    col("deal_count").desc()
)

print("Investor Deal Count:")
display(investor_deal_count)


# ============================================================
# 14. WRITE INVESTOR DEAL COUNT
# ============================================================

investor_deal_count.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(investor_count_path)

print("investor_deal_count written successfully.")


# ============================================================
# 15. AVERAGE DEAL BY STAGE
# ============================================================
#
# Business Question:
# What is the average deal size at Seed vs Series A vs Series B?
#
# SQL Technique:
# AVG() + CASE WHEN + GROUP BY
#
# PDF Requirement:
# avg_deal_by_stage
# ============================================================

print("Creating avg_deal_by_stage...")

stage_df = df.filter(
    col("investment_type").isNotNull()
    & (trim(col("investment_type")) != "")
)

stage_df = stage_df.withColumn(
    "funding_stage",
    when(
        lower(col("investment_type")).contains("seed"),
        "Seed"
    )
    .when(
        lower(col("investment_type")).contains("series a"),
        "Series A"
    )
    .when(
        lower(col("investment_type")).contains("series b"),
        "Series B"
    )
    .when(
        lower(col("investment_type")).contains("series c"),
        "Series C"
    )
    .when(
        lower(col("investment_type")).contains("series d"),
        "Series D"
    )
    .when(
        lower(col("investment_type")).contains("series e"),
        "Series E"
    )
    .when(
        lower(col("investment_type")).contains("series f"),
        "Series F"
    )
    .when(
        lower(col("investment_type")).contains("series g"),
        "Series G"
    )
    .when(
        lower(col("investment_type")).contains("debt"),
        "Debt"
    )
    .when(
        lower(col("investment_type")).contains("private equity"),
        "Private Equity"
    )
    .otherwise("Other")
)

avg_deal_by_stage = stage_df.groupBy(
    "funding_stage"
).agg(
    avg("amount_usd").alias("average_deal_usd"),
    count("*").alias("deal_count"),
    spark_sum("amount_usd").alias("total_funding_usd")
).withColumn(
    "average_deal_usd",
    round(col("average_deal_usd"), 2)
).withColumn(
    "total_funding_usd",
    round(col("total_funding_usd"), 2)
).orderBy(
    col("average_deal_usd").desc()
)

print("Average Deal By Stage:")
display(avg_deal_by_stage)


# ============================================================
# 16. WRITE AVERAGE DEAL BY STAGE
# ============================================================

avg_deal_by_stage.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(avg_stage_path)

print("avg_deal_by_stage written successfully.")


# ============================================================
# 17. GOLD VALIDATION
# ============================================================

print("==========================================")
print("GOLD LAYER VALIDATION")
print("==========================================")

print(
    "top_funded_sectors rows:",
    top_funded_sectors.count()
)

print(
    "city_funding_rank rows:",
    city_funding_rank.count()
)

print(
    "sector_yoy_snapshot rows:",
    sector_yoy_snapshot.count()
)

print(
    "investor_deal_count rows:",
    investor_deal_count.count()
)

print(
    "avg_deal_by_stage rows:",
    avg_deal_by_stage.count()
)

print("==========================================")
print("GOLD LAYER COMPLETED SUCCESSFULLY")
print("==========================================")


# ============================================================
# 18. DISPLAY ALL GOLD OUTPUTS
# ============================================================

print("========== TOP FUNDED SECTORS ==========")
display(
    spark.read
    .format("delta")
    .load(top_sectors_path)
)

print("========== CITY FUNDING RANK ==========")
display(
    spark.read
    .format("delta")
    .load(city_rank_path)
)

print("========== SECTOR YOY SNAPSHOT ==========")
display(
    spark.read
    .format("delta")
    .load(sector_yoy_path)
)

print("========== INVESTOR DEAL COUNT ==========")
display(
    spark.read
    .format("delta")
    .load(investor_count_path)
)

print("========== AVERAGE DEAL BY STAGE ==========")
display(
    spark.read
    .format("delta")
    .load(avg_stage_path)
)

Spark Session Started
Reading Silver Delta data...
Silver Row Count: 1100
Silver Columns:
['funding_date', 'funding_year', 'startup_name', 'industry_vertical', 'sub_vertical', 'city', 'investor_names', 'investment_type', 'amount_usd']
root
 |-- funding_date: date (nullable = true)
 |-- funding_year: integer (nullable = true)
 |-- startup_name: string (nullable = true)
 |-- industry_vertical: string (nullable = true)
 |-- sub_vertical: string (nullable = true)
 |-- city: string (nullable = true)
 |-- investor_names: string (nullable = true)
 |-- investment_type: string (nullable = true)
 |-- amount_usd: double (nullable = true)



funding_date,funding_year,startup_name,industry_vertical,sub_vertical,city,investor_names,investment_type,amount_usd
2020-02-26,2020,QuantumSolutions,Mobility,EV,Noida,IFC,Seed,104000.0
2021-01-24,2021,HyperLabs,Mobility,Ride Sharing,Chennai,Tiger Global,Angel,29000.0
2020-02-22,2020,Porter,Retail,E-Retail,Hyderabad,A91 Partners,Seed,1788000.0
2021-01-25,2021,AgriHive,EdTech,Coding Bootcamp,Noida,"Falcon Edge, IFC",Seed,157000.0
2024-01-16,2024,FreshBox,FoodTech,Food Delivery,Chennai,Tiger Global Management,Growth,1.54344E8
2023-10-17,2023,Dream11,Media,Content,Bengaluru,"Elevation Capital, Info Edge, Matrix Partners India",Pre-Series A,175000.0
2021-01-28,2021,AgriFit,Enterprise,Automation,Hyderabad,Elevation Capital,Series A,6512000.0
2022-05-08,2022,FoodSolutions,EdTech,Test Prep,Bengaluru,"IFC, Ventures India",Series C,1.14859E8
2024-02-09,2024,BlueSpace,Enterprise,Security,Kolkata,Blume Ventures,Series A,2359000.0
2020-01-26,2020,Bounce,AgriTech,Marketplace,Pune,Kalaari Capital,Growth,5.5675E8


Creating top_funded_sectors...
Top Funded Sectors:


industry_vertical,total_funding_usd,deal_count,average_deal_usd
FoodTech,3.477282E9,80,4.3466025E7
Consumer Electronics,2.703489E9,87,3.107458621E7
Retail,2.673074E9,89,3.003453933E7
Mobility,2.460096E9,86,2.860576744E7
Media,2.292449E9,85,2.696998824E7
AgriTech,2.142559E9,82,2.612876829E7
E-commerce,1.772544E9,90,1.969493333E7
FinTech,1.631375E9,73,2.234760274E7
SaaS,1.629119E9,72,2.262665278E7
EdTech,1.627226E9,78,2.086187179E7


top_funded_sectors written successfully.
Creating city_funding_rank...
City Funding Ranking:


...

industry_vertical,funding_year,total_funding_usd,previous_year_funding_usd,yoy_change_usd,yoy_change_percentage,deal_count,effective_from,effective_to,is_current,record_hash
AgriTech,2020,6.0977E8,null,null,null,16,2026-08-08,null,true,AgriTech||2020||6.0977E8||16
AgriTech,2021,2.96223E8,6.0977E8,-3.13547E8,-51.42,11,2026-08-08,null,true,AgriTech||2021||2.96223E8||6.0977E8||-3.13547E8||-51.42||11
AgriTech,2022,1.34158E8,2.96223E8,-1.62065E8,-54.71,18,2026-08-08,null,true,AgriTech||2022||1.34158E8||2.96223E8||-1.62065E8||-54.71||18
AgriTech,2023,1.91129E8,1.34158E8,5.6971E7,42.47,13,2026-08-08,null,true,AgriTech||2023||1.91129E8||1.34158E8||5.6971E7||42.47||13
AgriTech,2024,5.24078E8,1.91129E8,3.32949E8,174.2,17,2026-08-08,null,true,AgriTech||2024||5.24078E8||1.91129E8||3.32949E8||174.2||17
AgriTech,2025,3.87201E8,5.24078E8,-1.36877E8,-26.12,7,2026-08-08,null,true,AgriTech||2025||3.87201E8||5.24078E8||-1.36877E8||-26.12||7
Consumer Electronics,2020,1.82886E8,null,null,null,18,2026-08-08,null,true,Consumer Electronics||2020||1.82886E8||18
Consumer Electronics,2021,7.50328E8,1.82886E8,5.67442E8,310.27,21,2026-08-08,null,true,Consumer Electronics||2021||7.50328E8||1.82886E8||5.67442E8||310.27||21
Consumer Electronics,2022,1.57918E8,7.50328E8,-5.9241E8,-78.95,15,2026-08-08,null,true,Consumer Electronics||2022||1.57918E8||7.50328E8||-5.9241E8||-78.95||15
Consumer Electronics,2023,7.1029E7,1.57918E8,-8.6889E7,-55.02,12,2026-08-08,null,true,Consumer Electronics||2023||7.1029E7||1.57918E8||-8.6889E7||-55.02||12


========== INVESTOR DEAL COUNT ==========


investor,deal_count,total_investment_usd,average_deal_usd
Y Combinator,84,1.958843E9,2.331955952E7
Mirae Asset,83,1.657445E9,1.996921687E7
Info Edge,82,2.014404E9,2.456590244E7
Accel,80,2.05803E9,2.5725375E7
IFC,78,1.395912E9,1.789630769E7
Sequoia Capital India,77,1.988404E9,2.582342857E7
Kalaari Capital,77,2.758095E9,3.581941558E7
Prosus Ventures,75,1.959231E9,2.612308E7
Tiger Global Management,74,1.334572E9,1.803475676E7
Ribbit Capital,73,1.805262E9,2.472961644E7


========== AVERAGE DEAL BY STAGE ==========


funding_stage,average_deal_usd,deal_count,total_funding_usd
Private Equity,2.620466E8,5,1.310233E9
Series D,2.2835391667E8,24,5.480494E9
Series C,9.930678947E7,95,9.434145E9
Other,5.656E7,126,7.12656E9
Series B,2.906374545E7,110,3.197012E9
Debt,9344571.43,21,1.96236E8
Series A,3685396.83,315,1.1609E9
Seed,447185.64,404,1.80663E8
